# 06 · Clusters: heatmaps and sample groups from the top genes

**Run order:** 02 facility CSVs → 03 select samples → 04 batch correction → 05 differential expression → **06 (this notebook)**. Kernel: **R (WGCNA)**.
This is the equivalent of the repo's notebook 42 (heatmaps) and the sample-grouping part of 46b, built on your own cohort.

**Which batch correction is used where (the thing that is easy to mix up):**

| Notebook | Counts it uses | How batch is handled |
|---|---|---|
| 04 | raw counts → **ComBat-seq** counts | removes the facility shift, for **display and clustering** |
| 05 | **raw** counts | facility is a **term in the model** (the correct way to *test*) |
| **06 (this one)** | the **ComBat-seq counts from 04** | batch already removed, so clusters are not just "which facility" |

So: yes, batch correction was done, in notebook 04, and this notebook uses it. Notebook 05's statistics deliberately do not use the corrected counts.

**What this does.**
1. Takes the top genes from notebook 05 (adjusted p < 0.05, fold change ≥ 2) and the batch-corrected counts from 04 (log2 counts per million, each gene z-scored over the samples shown).
2. Draws a heatmap for each comparison, and one of all top genes over all samples. Samples and genes are grouped by hierarchical clustering (Ward's method, `ward.D2`, Euclidean distance, which is Minkowski with p = 2).
3. Cuts the sample tree into `K_CLUSTERS` groups and cross-tabulates each group against **diagnosis** and against **batch**. If a cluster is mostly one batch rather than one diagnosis, batch is still driving it.
4. If the repo's interferon gene lists are available, draws the interferon heatmap and prints an interferon score per diagnosis (healthy controls should be lowest).

Sample names carry patient initials, so heatmaps show **no sample names**; samples are identified by the coloured strips (diagnosis, batch).

**Reads:** `batch_correction/counts_combatseq_eligible.csv` (04), `rnaseq_samples_eligible.csv` (03), `differential_expression/` (05).
**Writes (all CSV, plots shown inline):** `rnaseq_out_v2/clusters/`: `sample_clusters.csv`, `genes_used_for_clustering.csv`, `ifn_score_by_diagnosis.csv`.

In [ ]:
RESULTS <- "/home/jovyan/session_data/Attempt0/endotype_results"
IN_DIR  <- file.path(RESULTS, "rnaseq_out_v2")
BC_DIR  <- file.path(IN_DIR, "batch_correction")
DE_DIR  <- file.path(IN_DIR, "differential_expression")
OUT_DIR <- file.path(IN_DIR, "clusters")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

K_CLUSTERS <- 4              # how many sample groups to cut the overall tree into
TOP_PER_COMPARISON <- 50     # genes shown in each comparison's heatmap (smallest adjusted p)
TOP_FOR_OVERALL <- 25        # genes taken from each comparison for the all-samples heatmap
RECODE <- c("Undifferentiated Interferonopathy" = "UAID")   # same merge as notebook 05
# Folder with ifn-response-genes-28.txt and nfkb-only-control-11-dejesus2020.txt (one gene symbol per line), from the repo:
GENE_SET_DIR <- "/home/jovyan/session_data/endotype-transcripts-007/genes"

for (f in c(file.path(BC_DIR, "counts_combatseq_eligible.csv"), file.path(DE_DIR, "summary_comparisons.csv")))
  if (!file.exists(f)) stop("Missing ", f, ": run the earlier notebook first (04 makes the corrected counts, 05 the top genes).")

if (!requireNamespace("edgeR", quietly = TRUE)) {            # one-time install if missing
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager", repos = "https://cloud.r-project.org")
  BiocManager::install("edgeR", ask = FALSE, update = FALSE)
}
suppressMessages(library(edgeR))

# ---- heatmap wrapper: older R versions accept only ONE colour strip; if two (diagnosis + batch) fail, draw the first only
.heatmap_base <- graphics::heatmap
heatmap <- function(..., RowSideColors = NULL) {
  a <- list(...)
  if (is.null(RowSideColors)) return(invisible(do.call(.heatmap_base, a)))
  r <- tryCatch(do.call(.heatmap_base, c(a, list(RowSideColors = RowSideColors))), error = function(e) e)
  if (inherits(r, "error")) {
    if (!is.matrix(RowSideColors)) stop(r)
    message("This R draws one colour strip only: showing the first (", colnames(RowSideColors)[1], "). The other strip is dropped.")
    r <- do.call(.heatmap_base, c(a, list(RowSideColors = unname(RowSideColors[, 1]))))
  }
  invisible(r)
}

## Read the corrected counts, the samples and the top genes
`cpm(..., log = TRUE)` after TMM normalization turns the ComBat-seq counts into log2 counts per million (the values the repo's heatmaps use). The top genes are the files `top_genes_<A>_vs_<B>.csv` of notebook 05, already sorted by adjusted p-value.

In [ ]:
cd <- read.csv(file.path(BC_DIR, "counts_combatseq_eligible.csv"), check.names = FALSE)
gene_id <- cd$Gene_ID; gene_name <- cd$Gene_Name
M <- as.matrix(cd[, !(colnames(cd) %in% c("Gene_ID", "Gene_Name"))]); storage.mode(M) <- "numeric"; rownames(M) <- gene_id
s <- read.csv(file.path(IN_DIR, "rnaseq_samples_eligible.csv"), check.names = FALSE, colClasses = "character")
stopifnot(!anyNA(match(colnames(M), s$sample_id)))
s <- s[match(colnames(M), s$sample_id), ]
dx <- s$dx_clean; hit <- dx %in% names(RECODE); dx[hit] <- unname(RECODE[dx[hit]]); s$diagnosis <- dx
id2name <- setNames(gene_name, gene_id)

logcpm <- cpm(calcNormFactors(DGEList(M)), log = TRUE)

summ <- read.csv(file.path(DE_DIR, "summary_comparisons.csv"), check.names = FALSE)
top <- lapply(setNames(summ$comparison, summ$comparison), function(k) {
  f <- file.path(DE_DIR, sprintf("top_genes_%s.csv", k)); if (file.exists(f)) read.csv(f) else NULL })
top <- top[sapply(top, function(t) !is.null(t) && nrow(t) > 0)]
print(table(diagnosis = s$diagnosis, batch = s$batch))
data.frame(comparison = names(top), top_genes = sapply(top, nrow), row.names = NULL)

## Heatmap function
Samples are rows, genes are columns. Each gene is z-scored over the samples shown (mean 0, standard deviation 1) and colours are capped at ±2.5, as in the repo's notebook 42. The two coloured strips on the left show diagnosis and batch; the legend is drawn once below.

In [ ]:
dx_lev <- sort(unique(s$diagnosis)); dx_col <- setNames(hcl.colors(length(dx_lev), "Dark 3"), dx_lev)
b_lev  <- sort(unique(s$batch));     b_col  <- setNames(hcl.colors(length(b_lev), "Set 2"), b_lev)

heat <- function(ids, samples_idx, title) {
  ids <- intersect(ids, rownames(logcpm))
  H <- scale(t(logcpm[ids, samples_idx, drop = FALSE]))            # samples x genes, each gene z-scored
  H <- H[, colSums(is.na(H)) == 0, drop = FALSE]
  if (ncol(H) < 2 || nrow(H) < 3) { cat("skipped (too few genes or samples):", title, "\n"); return(invisible(NULL)) }
  Hc <- pmin(pmax(H, -2.5), 2.5)
  hr <- hclust(dist(H), method = "ward.D2"); hc <- hclust(dist(t(H)), method = "ward.D2")
  m  <- s[samples_idx, ]
  side <- unname(dx_col[m$diagnosis])
  heatmap(Hc, Rowv = as.dendrogram(hr), Colv = as.dendrogram(hc), scale = "none", RowSideColors = side,
          labRow = rep("", nrow(Hc)), labCol = id2name[colnames(Hc)], cexCol = 0.55, margins = c(8, 2),
          col = hcl.colors(51, "Blue-Red 3"),
          main = sprintf("%s\n%d genes x %d samples; log2 CPM of ComBat-seq counts, z-scored; ward.D2", title, ncol(Hc), nrow(Hc)),
          cex.main = 0.8)
  invisible(list(rows = hr, cols = hc, genes = colnames(Hc), samples = samples_idx))
}
options(repr.plot.width = 12, repr.plot.height = 1.6)
par(mar = c(0, 0, 0, 0)); plot.new()
legend("topleft",  title = "diagnosis", legend = names(dx_col), fill = dx_col, bty = "n", ncol = 3, cex = 0.9)
legend("topright", title = "batch",     legend = names(b_col),  fill = b_col,  bty = "n", ncol = 2, cex = 0.9)
options(repr.plot.width = 12, repr.plot.height = 9)

## One heatmap per comparison
For each comparison of notebook 05, the top genes (up to `TOP_PER_COMPARISON`, smallest adjusted p) over the samples of its two diagnoses. A comparison with no top genes has no heatmap.

In [ ]:
safe <- function(x) gsub("[^A-Za-z0-9]+", "-", x)
for (k in names(top)) {
  parts <- strsplit(k, "_vs_")[[1]]
  idx <- which(safe(s$diagnosis) %in% parts)
  heat(head(top[[k]]$gene_id, TOP_PER_COMPARISON), idx, sprintf("%s: top genes", sub("_vs_", " against ", k)))
}

## All top genes, all samples: sample groups
The first `TOP_FOR_OVERALL` top genes of each comparison, over all eligible samples. The sample tree is cut into `K_CLUSTERS` groups. **Read the two tables:** a group that is mostly one diagnosis is a disease-driven cluster; a group that is mostly one batch means batch still shapes the data after correction.

In [ ]:
ids_all <- unique(unlist(lapply(top, function(t) head(t$gene_id, TOP_FOR_OVERALL))))
r <- heat(ids_all, seq_len(nrow(s)), "All top genes, all samples")
cl <- cutree(r$rows, K_CLUSTERS)
cat("\ncluster x diagnosis\n"); print(table(cluster = cl, diagnosis = s$diagnosis[r$samples]))
cat("\ncluster x batch\n");     print(table(cluster = cl, batch = s$batch[r$samples]))

## The interferon genes
The 28 interferon response genes and the 11 NF-kB-only control genes of the repo (Kim H et al. *J Interferon Cytokine Res* 2018;38:171-185; de Jesus AA et al. *J Clin Invest* 2020;130:1669-1682). The **interferon score** of a sample is the mean z-score of the 28 genes. In CANDLE, SAVI, AGS and UAID it should be higher than in healthy controls. If the gene-list files are not found, this part is skipped with a message.

In [ ]:
read_set <- function(f) { p <- file.path(GENE_SET_DIR, f); if (!file.exists(p)) return(NULL)
  x <- trimws(readLines(p)); x[x != "" & !startsWith(x, "#")] }
irg <- read_set("ifn-response-genes-28.txt"); nfk <- read_set("nfkb-only-control-11-dejesus2020.txt")
if (is.null(irg)) {
  cat("Gene lists not found in", GENE_SET_DIR, "- set GENE_SET_DIR to the repo's genes/ folder to run this part.\n")
} else {
  irg_id <- gene_id[match(irg, gene_name)]; irg_id <- irg_id[!is.na(irg_id)]
  nfk_id <- if (is.null(nfk)) character(0) else gene_id[match(nfk, gene_name)]; nfk_id <- nfk_id[!is.na(nfk_id)]
  cat("found:", length(irg_id), "of", length(irg), "interferon genes;", length(nfk_id), "NF-kB-only genes\n")
  Z <- scale(t(logcpm[irg_id, , drop = FALSE])); Z <- Z[, colSums(is.na(Z)) == 0, drop = FALSE]
  ifn_score <- rowMeans(Z)
  ifn_tab <- data.frame(diagnosis = names(tapply(ifn_score, s$diagnosis, median)),
                        samples = as.integer(table(s$diagnosis)[names(tapply(ifn_score, s$diagnosis, median))]),
                        median_ifn_score = round(as.numeric(tapply(ifn_score, s$diagnosis, median)), 2))
  print(ifn_tab[order(ifn_tab$median_ifn_score), ], row.names = FALSE)
  write.csv(ifn_tab, file.path(OUT_DIR, "ifn_score_by_diagnosis.csv"), row.names = FALSE)
  heat(c(irg_id, nfk_id), seq_len(nrow(s)), "28 interferon response genes + NF-kB-only controls, all samples")
}

## Save (all CSV)

In [ ]:
write.csv(data.frame(sample_id = s$sample_id[r$samples], patient_key = s$patient_key[r$samples], cluster = cl,
                     diagnosis = s$diagnosis[r$samples], batch = s$batch[r$samples]),
          file.path(OUT_DIR, "sample_clusters.csv"), row.names = FALSE)
write.csv(data.frame(gene_id = r$genes, gene_name = id2name[r$genes]),
          file.path(OUT_DIR, "genes_used_for_clustering.csv"), row.names = FALSE)
list.files(OUT_DIR)

---
## What to check
1. **Cluster × diagnosis** and **cluster × batch**: do the groups follow disease or facility?
2. **Interferon score by diagnosis**: healthy controls lowest, the interferonopathies higher. If not, the cohort or batch handling needs a second look.
3. **Healthy controls** sit with each other only if they are truly different from the patients; if they scatter among the patients, say so in your results.

## References
1. Ward JH. Hierarchical grouping to optimize an objective function. *J Am Stat Assoc* 1963;58:236-244.
2. Murtagh F, Legendre P. Ward's hierarchical agglomerative clustering method: which algorithms implement Ward's criterion? *J Classif* 2014;31:274-295.
3. Zhang Y, Parmigiani G, Johnson WE. ComBat-seq: batch effect adjustment for RNA-seq count data. *NAR Genom Bioinform* 2020;2:lqaa078.
4. Robinson MD, McCarthy DJ, Smyth GK. edgeR. *Bioinformatics* 2010;26:139-140.
5. Kim H et al. *J Interferon Cytokine Res* 2018;38:171-185. de Jesus AA et al. *J Clin Invest* 2020;130:1669-1682.